In [ ]:
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
sns.set_style('whitegrid')
from sklearn.metrics import confusion_matrix , classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Flatten , Conv2D , MaxPooling2D , Dropout , Activation , BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam , Adamax
from tensorflow.keras import regularizers

#Warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
#tf.keras.mixed_precision.set_global_policy('mixed_float16')

import tensorflow as tf
import numpy as np
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet121, ResNet50V2
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, ReLU, Add
from tensorflow.keras.models import Model
from tensorflow.keras.losses import KLDivergence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications import DenseNet169, MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D
import copy

import tensorflow as tf
from tensorflow.keras import layers
import tensorflow as tf
from tensorflow.keras import layers


## Run configuration

In [ ]:
# Experiment configuration. Set these BEFORE the original random resampling.
from pathlib import Path
from datetime import datetime
import time

SEED = 123
USE_CLOSED_LOOP = True     # False = fixed 0.5/0.5 baseline, same losses/metrics/schedule.
MAX_EPOCHS = 200           # Original MAIL maximum.
BATCH_SIZE = 32            # Original fit() default, now made explicit.
INITIAL_LEARNING_RATE = 1e-4
RUN_BINDING_CHECK = True   # One forward pass; no optimizer update.

# Fresh directory per run; keep separate results for fixed vs controlled runs.
RUN_TAG = "closed_loop" if USE_CLOSED_LOOP else "fixed_weights"
RUN_DIR = Path("mail_runs") / f"{RUN_TAG}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

tf.keras.utils.set_random_seed(SEED)

CONTROLLER_CONFIG = dict(
    head_pool=[[0.70, 0.30], [0.50, 0.50], [0.30, 0.70]],
    gamma=0.07,
    eta=0.5,
    ix_bias=0.05,
    target_accuracy=(0.88, 0.88),  # Starting setpoints, NOT validated optimal targets.
    Kp_head=1.5,
    Ki_head=0.3,
    Kd_head=0.0,                 # 0 means PI by default; nonzero enables D action.
    ctrl_lr_head=0.15,
    bias_clip_head=1.5,
    integral_clip=10.0,
    ema_decay=0.6,
    min_head_weight=0.15,        # Neither task can be removed from the objective.
    max_weight_step=0.10,       # Maximum absolute per-epoch change per head.
    warmup_epochs=3,            # Begin with three fixed 0.5/0.5 epochs.
    seed=SEED,
)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Run directory:", RUN_DIR.resolve())


## Data and source-preservation notes

The following original loading, independent splitting, augmentation, and random
subsampling cells are retained. Keep the original `.npy` files in the notebook's
working directory or edit their paths. This integration does not invent data,
patient IDs, or a new sampling scheme.

**Important:** the source independently subsamples the two inputs to equalize
row counts. Equal cardinalities do not establish patient-/sample-level paired
modalities. If your study requires genuine paired multimodal data, build the
alignment and leakage-safe splits from identifiers before training. If the
construction intentionally combines unpaired tasks, describe it as such. The
checks below validate shapes and labels, not identity alignment.


## Modality 1

In [ ]:
X_train_h = np.load('x_train.npy')
y_train_h = np.load('y_train.npy')
X_test_h = np.load('x_test(1).npy')
y_test_h = np.load('y_test(1).npy')

X_val_h = np.load('x_val.npy')
y_val_h = np.load('y_val.npy')


X_train_h.shape, y_train_h.shape, X_test_h.shape, y_test_h.shape, X_val_h.shape, y_val_h.shape


In [ ]:
random_indices = np.random.choice(1001, 810, replace=False)

X_test_h1 = X_test_h[random_indices]
y_test_h1 = y_test_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape

In [ ]:
random_indices = np.random.choice(1002, 648, replace=False)

X_val_h1 = X_val_h[random_indices]
y_val_h1 = y_val_h[random_indices]

X_test_h1.shape, y_test_h1.shape, X_test_h.shape, y_test_h.shape, X_val_h1.shape, y_val_h1.shape

In [ ]:
X_train_h.shape, y_train_h.shape, X_test_h.shape, y_test_h.shape

## Modality 2

In [ ]:
X_train_s = np.load('X.npy')
y_train_s = np.load('Y.npy')

X_train_s.shape, y_train_s.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(X_train_s, y_train_s, test_size=0.2, random_state=42)

X_train_s.shape,X_test_s.shape, y_train_s.shape,y_test_s.shape, y_val_s.shape,y_val_s.shape

In [ ]:
import numpy as np
import cv2

def rotate_image(image, angle):
    """
    Rotate the image by the specified angle.
    """
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_image = cv2.warpAffine(image, rotation_matrix, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return rotated_image

def translate_image(image, tx, ty):
    """
    Translate the image by the specified translation parameters.
    """
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    translated_image = cv2.warpAffine(image, translation_matrix, image.shape[1::-1])
    return translated_image

def apply_gaussian_blur(image, kernel_size=3):
    """
    Apply Gaussian Blur to the image to reduce noise and improve generalization.
    """
    blurred_image = cv2.GaussianBlur(image, (kernel_size, kernel_size), 0)
    return blurred_image

# Augmentation parameters
rotation_angles = [20]
translations = [(5, 5)]
kernel_sizes = [3]  # Gaussian Blur kernel sizes

augmented_X_train = []
augmented_y_train = []

for image, label in zip(X_train_s, y_train_s):
    # Augment with rotations
    for angle in rotation_angles:
        rotated_image = rotate_image(image, angle)
        augmented_X_train.append(rotated_image)
        augmented_y_train.append(label)

    # Augment with translations
    for tx, ty in translations:
        translated_image = translate_image(image, tx, ty)
        augmented_X_train.append(translated_image)
        augmented_y_train.append(label)

    # Augment with Gaussian Blur
    for kernel_size in kernel_sizes:
        blurred_image = apply_gaussian_blur(image, kernel_size)
        augmented_X_train.append(blurred_image)
        augmented_y_train.append(label)

# Convert lists to numpy arrays
augmented_X_train = np.array(augmented_X_train)
augmented_y_train = np.array(augmented_y_train)

# Shuffle the data
shuffle_indices = np.random.permutation(len(augmented_X_train))
augmented_X_train = augmented_X_train[shuffle_indices]
augmented_y_train = augmented_y_train[shuffle_indices]
augmented_X_train.shape, augmented_y_train.shape


In [ ]:
random_indices = np.random.choice(7773, 5421, replace=False)

augmented_X_train = augmented_X_train[random_indices]
augmented_y_train = augmented_y_train[random_indices]

augmented_X_train.shape, augmented_y_train.shape


In [ ]:
X_train_s = np.concatenate((X_train_s, augmented_X_train), axis=0)
y_train_s = np.concatenate((y_train_s, augmented_y_train), axis=0)
X_train_s.shape, y_train_s.shape

In [ ]:
X_train_s.shape, y_train_s.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers


def multi_kernel_groupwise_conv2(x, filters, groups=16, strides=1):
    # 1x1 Group-wise Convolution
    conv1x1 = layers.DepthwiseConv2D(kernel_size=1, strides=strides, padding="same")(x)

    # 3x3 Group-wise Convolution
    conv3x3 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same")(x)

    # 5x5 Group-wise Convolution
    conv5x5 = layers.DepthwiseConv2D(kernel_size=7, strides=strides, padding="same")(x)

    # Depthwise 3x3 Group-wise Convolution
    #depthwise3x3 = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x)
    
    # Concatenate all outputs along the channel axis
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")(x1)

    x = layers.Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(x)
    x = layers.Add()([x, x1])
    x = layers.Activation('relu')(x)
    
    return x

import tensorflow as tf
from tensorflow.keras import layers

class MultiKernelGroupwiseConv2(layers.Layer):
    def __init__(self, filters, groups=16, strides=1, **kwargs):
        super(MultiKernelGroupwiseConv2, self).__init__(**kwargs)
        self.filters = filters
        self.groups = groups
        self.strides = strides

        # 1x1 Group-wise Convolution
        self.conv1x1 = layers.DepthwiseConv2D(kernel_size=1, strides=strides, padding="same")

        # 3x3 Group-wise Convolution
        self.conv3x3 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same")

        # 5x5 Group-wise Convolution
        self.conv5x5 = layers.DepthwiseConv2D(kernel_size=7, strides=strides, padding="same")

        # Final 1x1 Group-wise Convolution
        self.final_conv = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same")

        # Shortcut Path
        self.shortcut_conv = layers.Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')

        # Activation Function
        self.activation = layers.Activation('relu')

    def call(self, inputs):
        conv1x1 = self.conv1x1(inputs)
        conv3x3 = self.conv3x3(inputs)
        conv5x5 = self.conv5x5(inputs)

        # Concatenate along the channel axis
        x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])
        x1 = self.final_conv(x1)

        # Shortcut connection
        x = self.shortcut_conv(inputs)
        x = layers.Add()([x, x1])
        x = self.activation(x)

        return x


# Custom Attention Block with Global Pooling
import tensorflow as tf
from tensorflow.keras import layers

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config




# Residual Attention Block (Using Spatial Attention)


import tensorflow as tf
from tensorflow.keras import layers

def multi_kernel_groupwise_conv1(x, filters, groups=8, strides=2):
    """ Advanced Multi-Kernel Groupwise Convolution Block (Without Attention, Optimized) """

    # Multi-Kernel Parallel Convolutions with Different Receptive Fields
    conv1x1 = layers.Conv2D(filters // 4, kernel_size=1, padding="same")(x)
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x)
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, padding="same")(x)
    conv_dilated = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=2, padding="same")(x)

    # Feature Fusion via Concatenation
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5, conv_dilated])

    # Channel Shuffle for Better Feature Mixing
    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)

    # Depthwise + Grouped Convolutions Hybrid
    x1 = layers.DepthwiseConv2D(kernel_size=3, padding="same")(x1)
    x1 = layers.Conv2D(filters, kernel_size=1, padding="same")(x1)

    # Downsampling x1
    x1 = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x1)

    # **Fix: Ensure x has the same number of channels as x1**
    x = layers.Conv2D(filters, kernel_size=1, padding="same")(x)
    x = layers.DepthwiseConv2D(kernel_size=3, strides=strides, padding="same")(x)

    # Residual Connection
    x = layers.Add()([x, x1])
    x = layers.Activation("relu")(x)
    
    return x

import tensorflow as tf
from tensorflow.keras import layers



import tensorflow as tf
from tensorflow.keras import layers

## MSGCA
'''def multi_kernel_groupwise_conv(x, filters, groups=8, strides=1, use_se=True):
    # GPC for MSGDC
    conv1x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, strides=strides, padding="same", use_bias=False)(x)
    

    # DDC (Multi-scale receptive fields) for MSGDC
    conv3x3 = layers.DepthwiseConv2D(kernel_size=3, dilation_rate=1, strides=strides, padding="same", use_bias=False)(x)
    

    # DWC (Multi-scale receptive fields) for MSGDC
    conv5x5 = layers.DepthwiseConv2D(kernel_size=5, strides=strides, padding="same", use_bias=False)(x)
    

    # Concatenation and 1x1 Fusion for MSHC
    x1 = layers.Concatenate()([conv1x1, conv3x3, conv5x5])

    def channel_shuffle(x, groups):
        batch, height, width, channels = tf.unstack(tf.shape(x))
        x = tf.reshape(x, [-1, height, width, groups, channels // groups])
        x = tf.transpose(x, [0, 1, 2, 4, 3])
        x = tf.reshape(x, [-1, height, width, channels])
        return x

    x1 = layers.Lambda(lambda x: channel_shuffle(x, groups))(x1)
    
    x1 = layers.Conv2D(filters, kernel_size=1, groups=groups, padding="same", use_bias=False)(x1)
    x1 = layers.BatchNormalization()(x1)

    # CA layer

    GMN = tf.reduce_min(inputs, axis=[1, 2])(x1)
    GMP = tf.reduce_max(inputs, axis=[1, 2])(x1)
    GAP = tf.reduce_mean(inputs, axis=[1, 2])(x1)
    GSP = tf.reduce_suminputs, axis=[1, 2])(x1)

    
    channel_add = GMN + GAP + GMP + GSP

    channel_sub = GMP - GAP - GMN
    
    channel_info = channel_add + channel_sub

    se = layers.Dense(filters // 16, activation='relu', use_bias=False)(div_channel)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)

    se = layers.Reshape((1, 1, filters))(se)
    
    self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='global_scale')
    se *= self.global_scale
    se = layers.Activation('sigmoid')(se)
    
    
    x1 = layers.Multiply()([x1, se])

    
    # Residual Connection
    x = layers.Conv2D(filters=filters, kernel_size=1, strides=strides, padding='same', use_bias=False)(x)
    

    x = layers.Add()([x, x1])
    x = layers.Activation('relu')(x)  # GELU activation for better convergence

    return x
'''




import tensorflow as tf
from tensorflow.keras import layers



def channel_shuffle(tensor, groups: int):
    """Shuffle channels (ShuffleNet-style)."""
    b, h, w, c = tf.unstack(tf.shape(tensor))
    tensor = tf.reshape(tensor, [b, h, w, groups, c // groups])
    tensor = tf.transpose(tensor, [0, 1, 2, 4, 3])
    tensor = tf.reshape(tensor, [b, h, w, c])
    return tensor


# EMILA
class MultiKernelGroupWiseConv(layers.Layer):
    """
    Multi-kernel / group-wise conv block with channel-shuffle,
    global min/max/avg/sum attention, and residual connection.
    """

    def __init__(self,
                 filters: int,
                 groups: int = 2,
                 strides: int = 1,
                 use_se: bool = True,
                 reduction: int = 4,
                 **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.groups = groups
        self.strides = strides
        self.use_se = use_se
        self.reduction = reduction

        # ── convolutions ───────────────────────────────────────────────
        # 1×1 grouped of MSGDC
        self.conv1x1 = layers.Conv2D(filters, 1,
                                     strides=strides,
                                     padding="same",
                                    # use_bias=False,
                                     groups=groups)
        self.bn1x1 = layers.BatchNormalization()

        # dw-3×3   of MSGDC
        self.dw3 = layers.DepthwiseConv2D(3, strides=strides,
                                          padding="same",
                                         # use_bias=False
                                         )
        self.bn_dw3 = layers.BatchNormalization()
        #self.pw3 = layers.Conv2D(filters, 1, padding="same", use_bias=False, groups=groups)
        #self.bn_pw3 = layers.BatchNormalization()

        # dw-5×5   of MSGDC
        self.dw5 = layers.DepthwiseConv2D(5, strides=strides,
                                          padding="same",
                                         # use_bias=False
                                         )
        self.bn_dw5 = layers.BatchNormalization()
        #self.pw5 = layers.Conv2D(filters, 1, padding="same", use_bias=False, groups=groups)
        #self.bn_pw5 = layers.BatchNormalization()

        # fusion after concat
        self.fuse = layers.Conv2D(filters, 1,
                                  padding="same",
                                 # use_bias=False, 
                                  groups=2
                                  )
        self.bn_fuse = layers.BatchNormalization()

        
        #if use_se:
        self.fc1 = layers.Dense(filters // reduction,
                                activation="relu")
        self.fc2 = layers.Dense(filters)
        self.reshape = layers.Reshape((1, 1, filters))

        # residual projection
        self.proj = layers.Conv2D(filters, 1,
                                  strides=strides,
                                  padding="same",
                                 # use_bias=False
                                 )
        self.bn_proj = layers.BatchNormalization()

        #self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer=tf.keras.initializers.HeNormal(), trainable=True, name='global_scale')
    

        self.act = layers.Activation("relu")

    # global trainable scale requested by you
    def build(self, input_shape):
        self.global_scale = self.add_weight(
            name="global_scale",
            shape=(1, 1, 1, 1),
            initializer=tf.keras.initializers.HeNormal(),
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        # ── branch 1: grouped 1×1 ──────────────────────────────────────
        b1 = self.conv1x1(x)

        # ── branch 2: dw-3×3 ──────────────────────────────────────────
        b2 = self.dw3(x)
        #b2 = self.bn_dw3(b2)
        #b2 = self.pw3(b2)

        # ── branch 3: dw-5×5 ──────────────────────────────────────────
        b3 = self.dw5(x)
        #b3 = self.bn_dw5(b3)
        #b3 = self.pw5(b3)

        # ── concat + shuffle + fuse ───────────────────────────────────
        out = tf.concat([b1, b2, b3], axis=-1)
        out = channel_shuffle(out, self.groups)
        out = self.bn_fuse(self.fuse(out))

        # ── CA  ────────────────
        
        g_min = tf.reduce_min(out, axis=[1, 2])
        g_max = tf.reduce_max(out, axis=[1, 2])
        g_avg = tf.reduce_mean(out, axis=[1, 2])
        g_sum = tf.reduce_sum(out, axis=[1, 2])

        channel_add = g_min + g_avg + g_max + g_sum 
        channel_sub = g_max - g_avg - g_min
        channel_info = channel_add + channel_sub

        se = self.fc1(channel_info)
        se = self.fc2(se)
        se = self.reshape(se)                  # (1,1,C)
        # ← the three lines you explicitly asked for

        
        se *= self.global_scale
        se = tf.nn.sigmoid(se)

        out *= se

        

        #print('out shape:', out.shape)
        #out = self.fuse(out) # GPC

        # ── residual connection ───────────────────────────────────────
        shortcut = self.bn_proj(self.proj(x))
        out = self.act(shortcut + out)
        return out



import tensorflow as tf
from tensorflow.keras import layers

### ERLA block
def RGSA(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x  # Save the original input for residual connection

    # Multi-Kernel Groupwise Convolution (First Layer)
    #x = MultiKernelGroupWiseConv(x, filters=filters, groups=16, strides=strides)

    x = MultiKernelGroupWiseConv(filters=filters, groups=16, strides=strides)(x)

    # Normalization and Activation
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Depthwise and Pointwise Convolution
    #x = MultiKernelGroupWiseConv(x, filters=filters, groups=16, strides=(1, 1))  # Strides=1 to avoid mismatch

    x = MultiKernelGroupWiseConv(filters=filters, groups=16, strides=(1, 1))(x)
    
    x = layers.Conv2D(filters, kernel_size=(1, 1), padding="same")(x)  # Pointwise conv for channel mixing
    x = layers.BatchNormalization()(x)

    # Adjust Shortcut if Needed (Ensure Matching Shape)
    if strides != (1, 1) or use_projection:
        shortcut = layers.Conv2D(filters, kernel_size=(1, 1), strides=strides, padding="same")(shortcut)  # Downsampling shortcut
        shortcut = layers.BatchNormalization()(shortcut)

    # Residual Connection (Ensure Same Shape)
    x = layers.Add()([x, shortcut])

    # Final Activation
    x = layers.Activation('relu')(x)

    return x


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

class GlobalMinPooling2D(layers.Layer):
    def __init__(self, **kwargs):
        super(GlobalMinPooling2D, self).__init__(**kwargs)

    def call(self, inputs):
        return tf.reduce_min(inputs, axis=[1, 2])

    def compute_output_shape(self, input_shape):
        return (input_shape[0], input_shape[-1])

    def get_config(self):
        config = super(GlobalMinPooling2D, self).get_config()
        return config


## EMCAM
class DeeperGlobalLocalAttentionLayer1(layers.Layer):
    def __init__(self, units, activation='sigmoid', dropout_rate=0.2, groups=16, use_scale=True, **kwargs):
        super(DeeperGlobalLocalAttentionLayer1, self).__init__(**kwargs)
        self.units = units
        self.activation = activation
        self.dropout_rate = dropout_rate
        self.use_scale = use_scale
        self.groups = groups

    def build(self, input_shape):
        input1, input2 = input_shape
        _, _, _, channels1 = input1
        _, _, _, channels2 = input2
        
        self.alpha1 = self.add_weight(shape=(1, 1, 1, channels1), initializer='ones', trainable=True)
        self.alpha2 = self.add_weight(shape=(1, 1, 1, channels2), initializer='ones', trainable=True)
        
        self.global_min_pooling = GlobalMinPooling2D()
        self.global_avg_pooling = layers.GlobalAveragePooling2D()
        self.global_max_pooling = layers.GlobalMaxPooling2D()
        
        self.global_attention = layers.Dense(units=self.units, activation=self.activation)
        
        self.local_conv1 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        self.local_conv2 = layers.Conv2D(filters=self.units, kernel_size=(1, 1), activation=self.activation)
        
        #self.min_pooling = MinPooling2D(pool_size = 1)
        #self.min_pooling1 = MinPooling2D(pool_size = 1)
        self.avg_pooling = layers.AveragePooling2D(pool_size = 1)
        self.max_pooling = layers.MaxPooling2D(pool_size = 1)
        
        self.sub = layers.Subtract()
        self.mod_scale1 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        self.mod_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        self.mod_scale3 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True)
        
        self.msgdc_conv1x1 = layers.Conv2D(
            self.units, 1, padding="same", use_bias=False, groups=self.groups
        )
        self.msgdc_dw3 = layers.DepthwiseConv2D(
            3, padding="same", use_bias=False
        )
        self.msgdc_dw5 = layers.DepthwiseConv2D(
            5, padding="same", use_bias=False
        )
        self.msgdc_fuse = layers.Conv2D(
            self.units, 1, padding="same", use_bias=False
        )
        
        if self.use_scale:
            self.global_scale = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale')
            self.local_scale = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale')
            
            self.global_scale2 = self.add_weight(shape=(1, 1, 1, 1), initializer='ones', trainable=True, name='global_scale2')
            self.local_scale2 = self.add_weight(shape=(1, 1, 1, self.units), initializer='ones', trainable=True, name='local_scale2')
        
        super(DeeperGlobalLocalAttentionLayer1, self).build(input_shape)

    def MFIFA(self, x):
            input_shape1 = tf.shape(x)
            flattened_inputs1 = tf.reshape(x, [-1, input_shape1[-1]])  # Flatten along the last axis
            dct_transformed1 = tf.signal.dct(flattened_inputs1, type=2, norm='ortho')
            dct_transformed1 = tf.reshape(dct_transformed1, input_shape1)  # Reshape back to original dimensions
            
            High1 = self.global_max_pooling(dct_transformed1)   ## High 1
            High1 = self.global_attention(High1)
            High1 = tf.expand_dims(tf.expand_dims(High1, 1), 1)
    
            Low1 = self.global_avg_pooling(dct_transformed1)  ## Low 1
            Low1 = self.global_attention(Low1)
            Low1 = tf.expand_dims(tf.expand_dims(Low1, 1), 1)
    
            
            Low2 = self.global_min_pooling(dct_transformed1)   ## Low 2
            Low2 = self.global_attention(Low2)
            Low2 = tf.expand_dims(tf.expand_dims(Low2, 1), 1)
            
            High2 = High1 - Low1
            
            High3 = dct_transformed1 - Low2
    
            High = High1 + High2 + High3
            Low = Low1 + Low2
            Mean = High - Low

            #High = tf.expand_dims(tf.expand_dims(High, 1), 1)
            #Low  = tf.expand_dims(tf.expand_dims(Low, 1), 1)
            #Mean = tf.expand_dims(tf.expand_dims(Mean, 1), 1)
    
            High *= self.mod_scale1 # Modulation
            Low *= self.mod_scale2 # Modulation
            Mean *= self.mod_scale3 # Modulation
            
            return High, Low, Mean

    
    def _multi_kernel_groupwise_conv(self, x):
        """MSGDC block, *re-using* layers defined in build()."""
        c1 = self.msgdc_conv1x1(x)
        c3 = self.msgdc_dw3(x)
        c5 = self.msgdc_dw5(x)

        concat = tf.concat([c1, c3, c5], axis=-1)
        return self.msgdc_fuse(concat)

        

    def call(self, inputs, training=None):
        input1, input2 = inputs
        
        # MFIFA

        High_1, Low_1, Mean_1 = self.MFIFA(input1) # for input modality 1
        High_2, Low_2, Mean_2 = self.MFIFA(input2) # for input modality 2

        
        High = High_1 + High_2
        Low = Low_1 + Low_2
        Mean = Mean_1 + Mean_2
        
        attention = tf.sigmoid(High + Low + Mean)  # MFIFA Attention map generation
        
        
        # Deeper Spatial Attention
        
        local_attention1 = self._multi_kernel_groupwise_conv(input1) #self.local_conv1(input1)
        local_attention1_mean = tf.reduce_mean(local_attention1, axis=[1, 2], keepdims=True)
        local_attention1_max = tf.reduce_max(local_attention1, axis=[1, 2], keepdims=True)
        local_attention1 = local_attention1_max + local_attention1_mean

        local_attention2 = self._multi_kernel_groupwise_conv(input2) #self.local_conv1(input1)
        local_attention2_mean = tf.reduce_mean(local_attention2, axis=[1, 2], keepdims=True)
        local_attention2_max = tf.reduce_max(local_attention2, axis=[1, 2], keepdims=True)
        local_attention2 = local_attention2_max + local_attention2_mean
        
        
        
        local_attention1_1 = self._multi_kernel_groupwise_conv(local_attention1)
        local_attention1_1_mean = tf.reduce_mean(local_attention1_1, axis=[1, 2], keepdims=True)
        local_attention1_1_max = tf.reduce_max(local_attention1_1, axis=[1, 2], keepdims=True)
        local_attention1 = local_attention1 + local_attention1_1_mean + local_attention1_1_max  + local_attention2 # Skip with Cross-modal Interactions

        
        
        local_attention2_1 = self._multi_kernel_groupwise_conv(local_attention2)
        local_attention2_1_mean = tf.reduce_mean(local_attention2_1, axis=[1, 2], keepdims=True)
        local_attention2_1_max = tf.reduce_max(local_attention2_1, axis=[1, 2], keepdims=True)
        local_attention2 = local_attention2 + local_attention2_1_mean + local_attention2_1_max  + local_attention1 # Skip with Cross-modal Interactions

        
        #local_attention2 = local_attention2_ + local_attention2__

        # Scale Global and Spatial Attention
    
        
        local_attention1 *= self.local_scale # Modulation
        local_attention2 *= self.local_scale2 # Modulation

        # Combine 
        attention1 = tf.sigmoid(local_attention1 + local_attention2) # EMSCA Attention map generation

        attention *= self.global_scale # Modulation
        attention1 *= self.local_scale # Modulation
        
        attention2 = tf.sigmoid(attention + attention1) # dUal domain Parallel Fusion Attention Map Generation via EMCAM
        
        
        return attention2 

    def get_config(self):
        config = super(DeeperGlobalLocalAttentionLayer1, self).get_config()
        config.update({'units': self.units, 'activation': self.activation, 'dropout_rate': self.dropout_rate,
                       'use_scale': self.use_scale})
        return config


## EMCAM Attention Map Integration Into Input modalities
class DeeperAttentionLayer1(layers.Layer):
    def __init__(self, units=64, use_scale=True, **kwargs):
        super(DeeperAttentionLayer1, self).__init__(**kwargs)
        self.units = units
        self.use_scale = use_scale

    def build(self, input_shape):
        input1, input2 = input_shape
        _, H, W, C1 = input1
        _, H, W, C2 = input2
        
        self.alpha3 = self.add_weight(shape=(1, 1, 1, C1), initializer='ones', trainable=True, name='alpha3')
        self.alpha4 = self.add_weight(shape=(1, 1, 1, C2), initializer='ones', trainable=True, name='alpha4')
        self.add = layers.Add()
        self.deeper_global_local_attention = DeeperGlobalLocalAttentionLayer1(units=self.units, activation='sigmoid', 
                                                                              dropout_rate=0.2,
                                                                              use_scale=self.use_scale)
        super(DeeperAttentionLayer1, self).build(input_shape)

    def call(self, inputs, training=None):
        input1, input2 = inputs
        attention2 = self.deeper_global_local_attention(inputs, training=training)
        attention_feature1 = input1 * attention2 * self.alpha3 # Attention Map Integration Into Input modality 1
        attention_feature2 = input2 * attention2 * self.alpha4 # Attention Map Integration Into Input modality 2
        
        #attention_feature = self.add([attention_feature1, attention_feature2])
        return attention_feature1, attention_feature2

    def get_config(self):
        config = super(DeeperAttentionLayer1, self).get_config()
        config.update({'units': self.units, 'use_scale': self.use_scale})
        return config


In [ ]:
## MSTL
def residual_GLC_branch1(inputs1, inputs2):
    
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs1)
    #x1 = DeeperAttentionLayer1(units=64, use_scale=True)(x1) ## MFA ####
    #x1 = AttentionBlock(64)(x1)
    x1 = BatchNormalization()(x1)
    x1 = tf.keras.layers.Activation('relu')(x1)
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)
    
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs2)
    #x2 = DeeperAttentionLayer1(units=64, use_scale=True)(x2) ## MFA ####
    #x2 = AttentionBlock(64)(x2)
    x2 = BatchNormalization()(x2)
    x2 = tf.keras.layers.Activation('relu')(x2)
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)
    
    ## ERLA
    x1 = RGSA(x1, filters=64)
    #
    x2 = RGSA(x2, filters=64)
    
    
    
    
    x1 = RGSA(x1, filters=64)
    
    x2 = RGSA(x2, filters=64)
    
    x1, x2 = DeeperAttentionLayer1(units=64, use_scale=True)([x1, x2])
    
    ## ERLA
    x1 = RGSA(x1, filters=128, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=128, strides=(2, 2), use_projection=True)
    
    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=128)
    
    x2 = RGSA(x2, filters=128)

    
    x1, x2 = DeeperAttentionLayer1(units=128, use_scale=True)([x1, x2])
    
    ## ERLA
    x1 = RGSA(x1, filters=256, strides=(2, 2), use_projection=True)
    
    
    x2 = RGSA(x2, filters=256, strides=(2, 2), use_projection=True)
    

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    
    x1 = RGSA(x1, filters=256)
    
    x2 = RGSA(x2, filters=256)
    
    x1, x2 = DeeperAttentionLayer1(units=256, use_scale=True)([x1, x2])
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512, strides=(2, 2), use_projection=True)
    
    x2 = RGSA(x2, filters=512, strides=(2, 2), use_projection=True)

    x1 = tf.keras.layers.Dropout(0.25)(x1, training = True) 
    x2 = tf.keras.layers.Dropout(0.25)(x2, training = True) 
    
    
    ## ERLA
    x1 = RGSA(x1, filters=512)
    x2 = RGSA(x2, filters=512)

    x1, x2 = DeeperAttentionLayer1(units=512, use_scale=True)([x1, x2])
    
    
    return x1, x2

In [ ]:
#def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)

import tensorflow.keras.layers as L

#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer

x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)


con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

#con = tf.keras.layers.Dropout(0.25)(con) 

x = GlobalAveragePooling2D()(con)
#print('GlobalAveragePooling2D x:',x.shape)

## TMTL
outputs1 = Dense(5, activation='softmax')(x)
outputs2 = Dense(7, activation='softmax')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])
#return model
print(model.summary())

## MAIL-specific adaptation of the supplied controller

For each epoch $t$, the training loss is

$$L_t = w_{s,t}\,\mathrm{CCE}(y_s,p_s) + w_{h,t}\,\mathrm{CCE}(y_h,p_h),
\qquad w_{s,t}+w_{h,t}=1.$$

This is **task-loss weighting**, not direct replacement of the learned feature-
fusion attention. Reweighting the losses changes the gradient reaching the
shared/fusion network through the ordinary Adam update.

The outer bandit selects a two-task template. The inner bounded, incremental
PID adjusts the template logits using smoothed, relative accuracy deficits.
Default derivative gain is zero, so its default operation is PI. The weight
floor and slew limit are added safeguards. There is no shallow/deep ordering
between these two task heads.

**Explicit differences from the pasted code:**

- The Dice/focal/BCE component mixture and 4-head monotone projection are not
  used; MAIL has only two classification CCE objectives.
- `target_dice` is replaced by two configurable classification accuracy targets.
  Error is centered across tasks, so the controller reallocates a fixed total
  weight; it cannot increase both task weights simultaneously.
- The gate-time PI is not used, since MAIL has no `ConcreteGate` penalty actuator.
- The pasted reward-style multiplicative update is replaced with an explicitly
  bounded-loss, log-space **EXP3-IX-style** update. This is a disclosed adaptation,
  not an unchanged copy or a transfer of a classical regret/convergence theorem.

For the bandit only, $z_t=\frac12\sum_{j\in\{s,h\}}\mathrm{CCE}_{j,t}/\log C_j$
accounts for the different class counts; the observed bounded loss is
$z_t/(1+z_t)$. Actual training losses are **not** divided by class count or
$\log C_j$. The selected arm receives
$\log W_a\leftarrow\log W_a-\eta\,[z_t/(1+z_t)]/(p_a+\xi)$.

Best-checkpoint selection, early stopping, and ReduceLROnPlateau use the fixed
`val_macro_ce = 0.5*(val_CCE_s + val_CCE_h)`, independent of the current control
weights. Here “macro” means the mean across the **two tasks**, not a class-
balanced metric. Weighted `val_loss` is logged but not used for those decisions.
Weights are assigned at epoch start, then remain unchanged through validation
and checkpointing. Only the next epoch receives the new control action.


In [ ]:
from __future__ import annotations

import copy
import json
from typing import Any, Sequence

import numpy as np


def _softmax64(values: Sequence[float]) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


class BanditPIDCore:
    

    def __init__(
        self,
        num_classes: Sequence[int] = (5, 7),
        head_pool: Sequence[Sequence[float]] | None = None,
        gamma: float = 0.07,
        eta: float = 0.5,
        ix_bias: float = 0.05,
        target_accuracy: float | Sequence[float] = 0.88,
        Kp_head: float = 1.5,
        Ki_head: float = 0.3,
        Kd_head: float = 0.0,
        ctrl_lr_head: float = 0.15,
        bias_clip_head: float = 1.5,
        integral_clip: float = 10.0,
        ema_decay: float = 0.6,
        min_head_weight: float = 0.15,
        max_weight_step: float = 0.10,
        warmup_epochs: int = 3,
        seed: int = 123,
        enabled: bool = True,
    ) -> None:
        classes = np.asarray(num_classes)
        if classes.ndim != 1 or classes.size != 2:
            raise ValueError('This MAIL adapter requires exactly two task heads.')
        if not np.all(np.isfinite(classes)) or np.any(classes < 2) or np.any(classes != np.floor(classes)):
            raise ValueError('num_classes must contain integers >= 2.')
        self.num_classes = classes.astype(np.int64)
        self.n_heads = 2
        default_pool = ((0.70, 0.30), (0.50, 0.50), (0.30, 0.70))
        pool = np.asarray(default_pool if head_pool is None else head_pool, dtype=np.float64)
        if pool.ndim != 2 or pool.shape[1] != self.n_heads or len(pool) < 1:
            raise ValueError('head_pool must have shape (number_of_arms, 2).')
        if not np.all(np.isfinite(pool)) or np.any(pool <= 0):
            raise ValueError('Every head_pool entry must be finite and strictly positive.')
        self.pool = pool / pool.sum(axis=1, keepdims=True)
        self.K = len(self.pool)
        scalar_values = [gamma, eta, ix_bias, Kp_head, Ki_head, Kd_head,
                         ctrl_lr_head, bias_clip_head, integral_clip, ema_decay,
                         min_head_weight, max_weight_step]
        if not np.all(np.isfinite(scalar_values)):
            raise ValueError('All controller hyperparameters must be finite.')
        if not 0 <= gamma < 1 or eta <= 0 or ix_bias <= 0:
            raise ValueError('Require 0 <= gamma < 1, eta > 0, ix_bias > 0.')
        if min(Kp_head, Ki_head, Kd_head, ctrl_lr_head) < 0:
            raise ValueError('PID gains and ctrl_lr_head must be nonnegative.')
        if bias_clip_head <= 0 or integral_clip <= 0:
            raise ValueError('PID clipping limits must be positive.')
        if not 0 <= ema_decay < 1:
            raise ValueError('ema_decay must lie in [0, 1).')
        if not 0 < min_head_weight < 0.5 or not 0 < max_weight_step <= 1:
            raise ValueError('Require 0 < min_head_weight < 0.5 and 0 < max_weight_step <= 1.')
        if int(warmup_epochs) != warmup_epochs or warmup_epochs < 0:
            raise ValueError('warmup_epochs must be a nonnegative integer.')
        targets = np.asarray(target_accuracy, dtype=np.float64)
        if targets.ndim == 0:
            targets = np.repeat(targets, self.n_heads)
        if targets.shape != (self.n_heads,) or not np.all(np.isfinite(targets)) or np.any((targets <= 0) | (targets > 1)):
            raise ValueError('target_accuracy must be a scalar or two values in (0, 1].')
        self.gamma, self.eta, self.ix_bias = map(float, (gamma, eta, ix_bias))
        self.targets = targets
        self.kp, self.ki, self.kd = map(float, (Kp_head, Ki_head, Kd_head))
        self.ctrl_lr = float(ctrl_lr_head)
        self.bias_clip = float(bias_clip_head)
        self.integral_clip = float(integral_clip)
        self.ema_decay = float(ema_decay)
        self.min_head_weight = float(min_head_weight)
        self.max_weight_step = float(max_weight_step)
        self.warmup_epochs, self.seed, self.enabled = int(warmup_epochs), int(seed), bool(enabled)
        self.rng = np.random.default_rng(self.seed)
        self.log_arm_weights = np.zeros(self.K, dtype=np.float64)
        self.bias = np.zeros(self.n_heads, dtype=np.float64)
        self.integral = np.zeros(self.n_heads, dtype=np.float64)
        self.prev_error = np.zeros(self.n_heads, dtype=np.float64)
        self.ema_accuracy: np.ndarray | None = None
        self.weights = np.full(self.n_heads, 0.5, dtype=np.float64)
        self.arm_probabilities = np.full(self.K, 1 / self.K)
        self.arm_index = -1
        self.epoch = -1
        self.completed_epoch = -1
        self.awaiting_observation = False
        self.config = dict(
            num_classes=self.num_classes.tolist(), head_pool=self.pool.tolist(),
            gamma=self.gamma, eta=self.eta, ix_bias=self.ix_bias,
            target_accuracy=self.targets.tolist(), Kp_head=self.kp, Ki_head=self.ki,
            Kd_head=self.kd, ctrl_lr_head=self.ctrl_lr, bias_clip_head=self.bias_clip,
            integral_clip=self.integral_clip, ema_decay=self.ema_decay,
            min_head_weight=self.min_head_weight, max_weight_step=self.max_weight_step,
            warmup_epochs=self.warmup_epochs, seed=self.seed, enabled=self.enabled,
        )

    def start_epoch(self, epoch: int) -> np.ndarray:
        if self.awaiting_observation:
            raise RuntimeError('observe() must finish the previous epoch before selecting another arm.')
        if int(epoch) != epoch or epoch < 0 or epoch <= self.completed_epoch:
            raise ValueError('epoch must be a nonnegative, strictly increasing integer.')
        self.epoch = int(epoch)
        self.awaiting_observation = True
        if not self.enabled or self.epoch < self.warmup_epochs:
            self.arm_index = -1
            self.weights = np.full(self.n_heads, 0.5, dtype=np.float64)
            return self.weights.copy()
        self.arm_probabilities = ((1.0 - self.gamma) * _softmax64(self.log_arm_weights)
                                  + self.gamma / self.K)
        self.arm_index = int(self.rng.choice(self.K, p=self.arm_probabilities))
        proposed = _softmax64(np.log(self.pool[self.arm_index]) + self.bias)
        # Strict task floor, without ranking the tasks as shallow/deep heads.
        proposed = self.min_head_weight + (1 - self.n_heads * self.min_head_weight) * proposed
        delta = proposed - self.weights
        largest_change = float(np.max(np.abs(delta)))
        if largest_change > self.max_weight_step:
            delta *= self.max_weight_step / largest_change
        self.weights += delta
        self.weights /= self.weights.sum()
        return self.weights.copy()

    def observe(self, val_ce: Sequence[float], val_accuracy: Sequence[float]) -> dict[str, float]:
        if not self.awaiting_observation:
            raise RuntimeError('Call start_epoch() exactly once before observe().')
        ce = np.asarray(val_ce, dtype=np.float64)
        accuracy = np.asarray(val_accuracy, dtype=np.float64)
        if ce.shape != (self.n_heads,) or accuracy.shape != (self.n_heads,):
            raise ValueError('Need two unweighted cross-entropies and two validation accuracies.')
        if not np.all(np.isfinite(ce)) or not np.all(np.isfinite(accuracy)):
            raise FloatingPointError('Nonfinite validation feedback; controller update aborted.')
        if np.any(ce < 0) or np.any((accuracy < 0) | (accuracy > 1)):
            raise ValueError('Cross-entropy must be nonnegative; accuracy must be in [0, 1].')
        if self.ema_accuracy is None:
            self.ema_accuracy = accuracy.copy()
        else:
            self.ema_accuracy = self.ema_decay * self.ema_accuracy + (1 - self.ema_decay) * accuracy
        # Normalize only the controller feedback, NOT the actual MAIL training loss.
        normalized_ce = float(np.mean(ce / np.log(self.num_classes)))
        bandit_loss = normalized_ce / (1.0 + normalized_ce)
        score = 1.0 / (1.0 + normalized_ce)
        error = np.maximum(self.targets - self.ema_accuracy, 0.0)
        error -= error.mean()  # redistribute a fixed total weight between tasks
        if self.enabled and self.arm_index >= 0:
            estimated_loss = bandit_loss / (self.arm_probabilities[self.arm_index] + self.ix_bias)
            self.log_arm_weights[self.arm_index] -= self.eta * estimated_loss
            # Log-space update prevents overflow in multiplicative weights.
            self.log_arm_weights -= np.max(self.log_arm_weights)
            derivative = error - self.prev_error
            candidate_integral = np.clip(self.integral + error, -self.integral_clip, self.integral_clip)
            update = self.ctrl_lr * (self.kp * error + self.ki * candidate_integral + self.kd * derivative)
            unconstrained = self.bias + update
            # Conditional integration: do not accumulate outward error at saturation.
            outward = ((unconstrained > self.bias_clip) & (error > 0)) | ((unconstrained < -self.bias_clip) & (error < 0))
            candidate_integral = np.where(outward, self.integral, candidate_integral)
            update = self.ctrl_lr * (self.kp * error + self.ki * candidate_integral + self.kd * derivative)
            self.integral = candidate_integral
            self.bias = np.clip(self.bias + update, -self.bias_clip, self.bias_clip)
            self.prev_error = error.copy()
        self.completed_epoch = self.epoch
        self.awaiting_observation = False
        return {
            'val_macro_ce': float(ce.mean()),
            'val_macro_accuracy': float(accuracy.mean()),
            'val_macro_normalized_ce': normalized_ce,
            'ctrl_score': score,
            'ctrl_bandit_loss': bandit_loss,
            'ctrl_arm': float(self.arm_index),
            'ctrl_arm_probability': float(self.arm_probabilities[self.arm_index]) if self.arm_index >= 0 else 1.0,
            'ctrl_active': float(self.enabled and self.arm_index >= 0),
            'ctrl_weight_s': float(self.weights[0]),
            'ctrl_weight_h': float(self.weights[1]),
            'ctrl_error_s': float(error[0]),
            'ctrl_error_h': float(error[1]),
            'ctrl_next_bias_s': float(self.bias[0]),
            'ctrl_next_bias_h': float(self.bias[1]),
        }

    def state_dict(self) -> dict[str, Any]:
        """Controller state only; this does NOT contain neural-network/optimizer state."""
        return {
            'version': 1, 'config': copy.deepcopy(self.config),
            'log_arm_weights': self.log_arm_weights.tolist(),
            'bias': self.bias.tolist(), 'integral': self.integral.tolist(),
            'prev_error': self.prev_error.tolist(),
            'ema_accuracy': None if self.ema_accuracy is None else self.ema_accuracy.tolist(),
            'weights': self.weights.tolist(),
            'arm_probabilities': self.arm_probabilities.tolist(),
            'arm_index': self.arm_index, 'epoch': self.epoch,
            'completed_epoch': self.completed_epoch,
            'awaiting_observation': self.awaiting_observation,
            'rng_state': copy.deepcopy(self.rng.bit_generator.state),
        }

    def load_state_dict(self, state: dict[str, Any]) -> None:
        """Restore controller-only state when all controller configuration matches."""
        if state.get('version') != 1 or json.dumps(state.get('config'), sort_keys=True) != json.dumps(self.config, sort_keys=True):
            raise ValueError('Controller checkpoint version/configuration does not match.')
        array_names = ('log_arm_weights', 'bias', 'integral', 'prev_error', 'weights', 'arm_probabilities')
        restored = {}
        for name in array_names:
            value = np.asarray(state[name], dtype=np.float64)
            if value.shape != getattr(self, name).shape or not np.all(np.isfinite(value)):
                raise ValueError(f'Invalid saved controller array: {name}')
            restored[name] = value
        if not np.isclose(restored['weights'].sum(), 1.0) or np.any(restored['weights'] <= 0):
            raise ValueError('Invalid saved task weights.')
        if not np.isclose(restored['arm_probabilities'].sum(), 1.0) or np.any(restored['arm_probabilities'] <= 0):
            raise ValueError('Invalid saved arm probabilities.')
        ema = state['ema_accuracy']
        if ema is not None:
            ema = np.asarray(ema, dtype=np.float64)
            if ema.shape != (self.n_heads,) or not np.all(np.isfinite(ema)) or np.any((ema < 0) | (ema > 1)):
                raise ValueError('Invalid saved EMA accuracy.')
        rng = np.random.default_rng()
        rng.bit_generator.state = copy.deepcopy(state['rng_state'])
        arm = int(state['arm_index'])
        if arm < -1 or arm >= self.K:
            raise ValueError('Invalid saved arm index.')
        for name, value in restored.items():
            setattr(self, name, value.copy())
        self.ema_accuracy = None if ema is None else ema.copy()
        self.arm_index = arm
        self.epoch, self.completed_epoch = int(state['epoch']), int(state['completed_epoch'])
        self.awaiting_observation = bool(state['awaiting_observation'])
        self.rng = rng


In [ ]:
from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Any, Sequence

import numpy as np
import tensorflow as tf



def _dynamic_head_cce(head_logits: Any, index: int):
    
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        per_example = tf.keras.losses.categorical_crossentropy(y_true, y_pred, from_logits=False)
        weight = tf.nn.softmax(tf.cast(head_logits, tf.float32))[index]
        return weight * per_example
    loss.__name__ = f'dynamic_head_{index + 1}_cce'
    return loss


def compile_mail_with_dynamic_weights(model, learning_rate: float = 1e-4, optimizer=None):
    
    if len(model.outputs) != 2 or len(model.output_names) != 2:
        raise ValueError('MAIL must have two output heads, ordered [task_s, task_h].')
    if hasattr(model, 'mail_head_logits'):
        raise ValueError('Dynamic weights are already attached. Rebuild MAIL for a fresh run.')
    classes = [int(output.shape[-1]) for output in model.outputs]
    if any(c < 2 for c in classes):
        raise ValueError('Each MAIL output must be a multiclass probability vector.')
    if not np.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError('learning_rate must be finite and positive.')
    # add_weight(), unlike an untracked external tf.Variable, includes the
    # controller's effective loss logits in best/last model-weight checkpoints.
    model.mail_head_logits = model.add_weight(
        name='mail_head_logits', shape=(2,), initializer='zeros',
        dtype='float32', trainable=False,
    )
    losses = [_dynamic_head_cce(model.mail_head_logits, j) for j in range(2)]
    metrics = [[tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
                tf.keras.metrics.CategoricalCrossentropy(name='unweighted_ce')]
               for _ in range(2)]
    if optimizer is None:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss=losses, metrics=metrics,
                  run_eagerly=False, jit_compile=False)
    return model.mail_head_logits


class ControlBanditPIDMAIL(tf.keras.callbacks.Callback):
    
    def __init__(self, head_logits_var, output_names: Sequence[str],
                 num_classes: Sequence[int] = (5, 7), verbose: int = 1, **core_kwargs):
        super().__init__()
        self.head_logits = head_logits_var
        self.output_names = tuple(output_names)
        if len(self.output_names) != 2 or len(set(self.output_names)) != 2:
            raise ValueError('Need two distinct output_names in [task_s, task_h] order.')
        self.core = BanditPIDCore(num_classes=num_classes, **core_kwargs)
        self.verbose = int(verbose)
        self._epoch_start = None
        self._fit_start = None
        self.last_feedback = {}

    def on_train_begin(self, logs=None):
        if getattr(self.model, 'mail_head_logits', None) is not self.head_logits:
            raise ValueError('Controller must use this model\'s tracked mail_head_logits.')
        if tuple(self.model.output_names) != self.output_names:
            raise ValueError('Controller output names do not match the compiled model.')
        self._fit_start = time.perf_counter()

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.perf_counter()
        weights = self.core.start_epoch(epoch)
        # Assign BEFORE training and keep unchanged through validation/checkpoint.
        self.head_logits.assign(np.log(weights).astype(np.float32))
        if self.verbose:
            print(f'\n[MAIL control] epoch={epoch + 1} arm={self.core.arm_index} '
                  f'weights(s,h)=({weights[0]:.4f}, {weights[1]:.4f})')

    def _read_metric(self, logs, name):
        if name not in logs:
            raise KeyError(f'Missing required metric {name!r}. '
                           f'Available keys: {sorted(logs)}. '
                           'Compile with compile_mail_with_dynamic_weights() and validate every epoch.')
        value = float(logs[name])
        if not math.isfinite(value):
            raise FloatingPointError(f'Nonfinite feedback metric: {name}={value}')
        return value

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            raise ValueError('Epoch logs are required for closed-loop feedback.')
        val_ce = [self._read_metric(logs, f'val_{name}_unweighted_ce') for name in self.output_names]
        val_acc = [self._read_metric(logs, f'val_{name}_accuracy') for name in self.output_names]
        feedback = self.core.observe(val_ce, val_acc)
        train_keys = [f'{name}_unweighted_ce' for name in self.output_names]
        if all(key in logs for key in train_keys):
            feedback['macro_ce'] = float(np.mean([float(logs[key]) for key in train_keys]))
        now = time.perf_counter()
        feedback['epoch_seconds_train_val'] = now - self._epoch_start
        feedback['fit_seconds_so_far'] = now - self._fit_start
        # Mutate the existing dict, even if it was empty; never use `logs or {}`.
        logs.update(feedback)
        self.last_feedback = feedback.copy()
        # IMPORTANT: do NOT reassign loss logits here. The next PID/bandit
        # action is applied in the next on_epoch_begin(), not to this checkpoint.

    def state_dict(self):
        return dict(core=self.core.state_dict(), output_names=list(self.output_names),
                    effective_head_logits=np.asarray(self.head_logits.numpy()).tolist(),
                    last_feedback=self.last_feedback,
                    note='Controller-only state. Complete optimizer/scheduler/data-RNG resume is not implemented.')


class ControllerStateWriter(tf.keras.callbacks.Callback):
    
    def __init__(self, controller: ControlBanditPIDMAIL, directory):
        super().__init__()
        self.controller = controller
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.best = float('inf')

    def _write(self, filename, state):
        target = self.directory / filename
        temporary = target.with_suffix(target.suffix + '.tmp')
        temporary.write_text(json.dumps(state, indent=2, allow_nan=False), encoding='utf-8')
        temporary.replace(target)

    def on_epoch_end(self, epoch, logs=None):
        logs = {} if logs is None else logs
        score = float(logs['val_macro_ce'])
        state = self.controller.state_dict()
        state['saved_after_epoch'] = int(epoch) + 1
        self._write('controller_last.json', state)
        if score < self.best:
            self.best = score
            self._write('controller_best.json', state)


def make_mail_callbacks(model, run_dir, enabled: bool = True,
                        controller_kwargs: dict | None = None,
                        early_stopping_patience: int = 80,
                        lr_patience: int = 70,
                        verbose: int = 1):
    
    if not hasattr(model, 'mail_head_logits'):
        raise ValueError('Call compile_mail_with_dynamic_weights(model) first.')
    if early_stopping_patience < 1 or lr_patience < 1:
        raise ValueError('Scheduler/stopping patience must be >= 1.')
    run_dir = Path(run_dir)
    # Separate checkpoint/diagnostic directories avoid competing callback paths.
    for subdir in ('best', 'last', 'diagnostics'):
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    options = dict(controller_kwargs or {})
    if 'enabled' in options or 'num_classes' in options:
        raise ValueError('Pass enabled separately; num_classes is inferred from MAIL.')
    controller = ControlBanditPIDMAIL(
        model.mail_head_logits, model.output_names,
        num_classes=[int(output.shape[-1]) for output in model.outputs],
        enabled=enabled, verbose=verbose, **options,
    )
    callbacks = [
        controller,
        tf.keras.callbacks.ModelCheckpoint(
            str(run_dir / 'best' / 'mail_best.weights.h5'),
            monitor='val_macro_ce', mode='min', save_best_only=True,
            save_weights_only=True, verbose=verbose,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            str(run_dir / 'last' / 'mail_last.weights.h5'),
            save_best_only=False, save_weights_only=True, verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_macro_ce', mode='min', factor=0.1,
            patience=lr_patience, min_lr=1e-5, verbose=verbose,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_macro_ce', mode='min', patience=early_stopping_patience,
            restore_best_weights=False, verbose=verbose,
        ),
        tf.keras.callbacks.CSVLogger(str(run_dir / 'diagnostics' / 'training.csv')),
        ControllerStateWriter(controller, run_dir / 'diagnostics'),
    ]
    return callbacks, controller


def validate_mail_arrays(x_values, y_values, split: str, model,
                         scan_images: int = 32) -> None:
    
    if len(x_values) != 2 or len(y_values) != 2:
        raise ValueError(f'{split}: expected two inputs and two labels.')
    lengths = [len(array) for array in [*x_values, *y_values]]
    if len(set(lengths)) != 1 or min(lengths) == 0:
        raise ValueError(f'{split}: input/target row counts must match and be >0; got {lengths}.')
    for j, (images, labels) in enumerate(zip(x_values, y_values)):
        images, labels = np.asarray(images), np.asarray(labels)
        expected = tuple(int(dim) for dim in model.inputs[j].shape[1:])
        if images.ndim != 4 or tuple(images.shape[1:]) != expected:
            raise ValueError(f'{split} input {j}: expected (N, {expected}), got {images.shape}.')
        if images.dtype.kind not in 'fiu' or not np.all(np.isfinite(images[:scan_images])):
            raise ValueError(f'{split} input {j}: invalid image dtype/nonfinite sampled values.')
        classes = int(model.outputs[j].shape[-1])
        if labels.shape != (lengths[0], classes):
            raise ValueError(f'{split} target {j}: expected one-hot/soft labels (N,{classes}); got {labels.shape}.')
        if labels.dtype.kind not in 'fiub' or not np.all(np.isfinite(labels)):
            raise ValueError(f'{split} target {j}: invalid label dtype/nonfinite values.')
        if np.any(labels < 0) or np.any(labels > 1) or not np.allclose(labels.sum(axis=1), 1.0, atol=1e-5):
            raise ValueError(f'{split} target {j}: each label row must be a probability vector summing to 1.')
    print(f'{split}: {lengths[0]} samples; array shapes/labels validated. Pair identity is NOT verified.')


def add_fixed_evaluation_metrics(results: dict, output_names: Sequence[str], num_classes=(5, 7)) -> dict:
    
    result = dict(results)
    ce = np.asarray([result[f'{name}_unweighted_ce'] for name in output_names], dtype=float)
    acc = np.asarray([result[f'{name}_accuracy'] for name in output_names], dtype=float)
    result['macro_ce'] = float(np.mean(ce))
    result['macro_accuracy'] = float(np.mean(acc))
    result['macro_normalized_ce'] = float(np.mean(ce / np.log(num_classes)))
    return result


def assert_dynamic_weight_binding(model, x_batch, y_batch) -> dict:
    
    predictions = model(x_batch, training=False)
    if len(predictions) != 2 or len(y_batch) != 2:
        raise ValueError('The binding check expects two predictions and target arrays.')
    targets = [tf.convert_to_tensor(y, dtype=tf.float32) for y in y_batch]
    predictions = [tf.stop_gradient(tf.cast(p, tf.float32)) for p in predictions]
    cce = [tf.reduce_mean(tf.keras.losses.categorical_crossentropy(y, p))
           for y, p in zip(targets, predictions)]

    @tf.function
    def traced_loss_terms():
        return tf.stack([tf.reduce_mean(model.loss[j](targets[j], predictions[j]))
                         for j in range(2)])

    saved = np.asarray(model.mail_head_logits.numpy()).copy()
    raw = np.asarray([float(value.numpy()) for value in cce])
    observed = []
    try:
        for weights in ([0.8, 0.2], [0.2, 0.8]):
            model.mail_head_logits.assign(np.log(weights).astype(np.float32))
            actual = np.asarray(traced_loss_terms().numpy())
            np.testing.assert_allclose(actual, raw * np.asarray(weights), rtol=2e-5, atol=2e-6)
            observed.append(actual.tolist())
    finally:
        model.mail_head_logits.assign(saved)
    if not np.all(np.isfinite(raw)):
        raise FloatingPointError('MAIL produced nonfinite cross-entropy during the binding check.')
    print('PASS: assigned head weights are read dynamically inside one traced loss graph.')
    return {'unweighted_ce': raw.tolist(), 'weighted_terms_80_20': observed[0],
            'weighted_terms_20_80': observed[1], 'optimizer_steps_performed': 0}


## Compile MAIL and attach closed-loop callbacks

In [ ]:
# Attach tracked dynamic coefficients and compile the existing MAIL model.
# The architecture/model object and head order are not replaced.
compile_mail_with_dynamic_weights(model, learning_rate=INITIAL_LEARNING_RATE)

callbacks, closed_loop_controller = make_mail_callbacks(
    model,
    run_dir=RUN_DIR,
    enabled=USE_CLOSED_LOOP,
    controller_kwargs=CONTROLLER_CONFIG,
    early_stopping_patience=80,  # Actual patience used in the original function.
    lr_patience=70,              # Original patience, factor 0.1, minimum LR 1e-5.
    verbose=1,
)

print("Model outputs [task_s, task_h]:", list(model.output_names))
print("Class counts:", [int(output.shape[-1]) for output in model.outputs])
print("Initial loss weights:", tf.nn.softmax(model.mail_head_logits).numpy())
print("Selection metric: val_macro_ce (fixed, equal-weight unweighted CE)")

run_config = {
    "seed": SEED,
    "use_closed_loop": USE_CLOSED_LOOP,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "initial_learning_rate": INITIAL_LEARNING_RATE,
    "tensorflow_version": tf.__version__,
    "model_output_names": list(model.output_names),
    "controller": closed_loop_controller.core.config,
    "selection_monitor": "val_macro_ce",
    "source_architecture_modified": False,
    "source_data_alignment_modified": False,
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")


## Pre-training shape and TensorFlow graph-binding checks

In [ ]:
# Keep the exact source ordering: s -> 5 classes; h -> 7 classes.
train_inputs = [X_train_s, X_train_h]
train_targets = [y_train_s, y_train_h]
validation_inputs = [X_val_s, X_val_h1]
validation_targets = [y_val_s, y_val_h1]
test_inputs = [X_test_s, X_test_h1]
test_targets = [y_test_s, y_test_h1]

validate_mail_arrays(train_inputs, train_targets, "train", model)
validate_mail_arrays(validation_inputs, validation_targets, "validation", model)
validate_mail_arrays(test_inputs, test_targets, "test", model)

if RUN_BINDING_CHECK:
    binding_report = assert_dynamic_weight_binding(
        model,
        [array[:2] for array in train_inputs],
        [array[:2] for array in train_targets],
    )
    (RUN_DIR / "loss_binding_check.json").write_text(
        json.dumps(binding_report, indent=2), encoding="utf-8"
    )


## Train

In [ ]:
# Same inputs/targets, maximum epochs, shuffle setting and default batch size
# as the source notebook. The callback is already connected to the live losses.
fit_start = time.perf_counter()
history = model.fit(
    train_inputs,
    train_targets,
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(validation_inputs, validation_targets),
    validation_freq=1,
    verbose=1,
    shuffle=True,
    callbacks=callbacks,
)
fit_wall_seconds = time.perf_counter() - fit_start

best_epoch = int(np.argmin(history.history["val_macro_ce"])) + 1
training_summary = {
    "completed_epochs": len(history.history["loss"]),
    "fit_wall_seconds": float(fit_wall_seconds),
    "best_epoch_by_fixed_macro_ce": best_epoch,
    "best_val_macro_ce": float(min(history.history["val_macro_ce"])),
    "best_checkpoint": str(RUN_DIR / "best" / "mail_best.weights.h5"),
    "last_checkpoint": str(RUN_DIR / "last" / "mail_last.weights.h5"),
}
(RUN_DIR / "training_summary.json").write_text(
    json.dumps(training_summary, indent=2), encoding="utf-8"
)
print(json.dumps(training_summary, indent=2))


## Evaluate the best checkpoint

In [ ]:
# Explicit best-weight reload works whether training stops early or hits 200.
# The chronological last checkpoint remains untouched in its separate folder.
best_weights_path = RUN_DIR / "best" / "mail_best.weights.h5"
if not best_weights_path.is_file():
    raise FileNotFoundError(f"No completed best checkpoint: {best_weights_path}")
model.load_weights(str(best_weights_path))

test_results = model.evaluate(
    test_inputs, test_targets, batch_size=BATCH_SIZE,
    return_dict=True, verbose=1,
)
test_results = add_fixed_evaluation_metrics(
    test_results, model.output_names,
    num_classes=[int(output.shape[-1]) for output in model.outputs],
)
(RUN_DIR / "test_results.json").write_text(
    json.dumps({name: float(value) for name, value in test_results.items()}, indent=2),
    encoding="utf-8",
)
print(json.dumps(test_results, indent=2))


## Inspect optimization and compare against the fixed baseline

Each run writes best/last `.weights.h5` files, `run_config.json`, a training CSV,
controller JSON diagnostics, a training summary, and test results. Model weights
must be loaded into a rebuilt compatible MAIL model; this notebook does not
claim arbitrary full-model `.keras` deserialization or complete interrupted-run
resumption of optimizer, scheduler, callback and data-RNG state.

To evaluate a speedup, use a fresh kernel and the same seed/data preparation/
architecture/batch size/scheduler, changing only `USE_CLOSED_LOOP` to `False`.
Compare the same validation-performance threshold and both tasks' metrics over
multiple seeds. Fewer early-stopped epochs alone do not establish faster
convergence. Avoid choosing thresholds or controller settings from test results.

`fit_wall_seconds` includes fit-time training, validation and callback work.
`epoch_seconds_train_val` ends before checkpoint/file-writing callbacks and is
not full end-to-end runtime. Preloading/augmentation, model construction, the
pre-fit binding check, and final evaluation are outside the fit timer.

The six original `Dropout(...)(x, training=True)` calls are preserved, not silently
changed by this integration. Review that explicit training override when
interpreting validation behavior or comparing Keras versions. The callback
cannot repair an underlying data-alignment or model-evaluation issue.


In [ ]:
# Show actual per-epoch control behavior, without another forward/validation pass.
import pandas as pd

training_log = pd.read_csv(RUN_DIR / "diagnostics" / "training.csv")
columns = [name for name in [
    "epoch", "loss", "val_loss", "val_macro_ce", "val_macro_accuracy",
    "ctrl_weight_s", "ctrl_weight_h", "ctrl_arm", "ctrl_active",
    "epoch_seconds_train_val", "fit_seconds_so_far",
] if name in training_log.columns]
training_log[columns].tail(15)
